# KYC setup – schema, tables, UC functions, and demo document images

**Run this notebook once** (or when you change catalog/schema) before using the KYC agent. It:
1. Creates the catalog (if not exists), schema, and the agent volume.
2. Creates and populates tables: `pep_screening`, `sanctions_screening`, `sanctioned_countries`, `fraud_list`, `accounts`, `sbi_risk_codes`.
3. Creates Unity Catalog functions: `check_pep`, `check_sanctions`, `check_sanctioned_country`, `check_customer_risk`, `check_fraud`.
4. Generates synthetic passport and utility bill images (Pillow) and uploads them to the agent volume.


In [ ]:
%run ../_shared/config

In [ ]:
%pip install pillow --quiet

In [ ]:
import os
from pyspark.sql.types import StructType, StructField, LongType, StringType, IntegerType
from pyspark.sql import Row

# CATALOG, SCHEMA, CATALOG_SCHEMA, VOLUME_PATH come from %run ../_shared/config
LLM_ENDPOINT = os.getenv("LLM_ENDPOINT", "databricks-meta-llama-3-3-70b-instruct")

print(f"Using CATALOG_SCHEMA={CATALOG_SCHEMA}, VOLUME_PATH={VOLUME_PATH}")

## 1. Create schema and tables

In [ ]:
# Create catalog and schema if they don't exist (CATALOG_SCHEMA e.g. main.kyc_demo)
spark.sql(f"CREATE SCHEMA IF NOT EXISTS {CATALOG_SCHEMA}")

# Create the agent volume if it doesn't exist (path e.g. /Volumes/main/kyc_demo/agent_volume)
vol_parts = VOLUME_PATH.strip("/").split("/")
if len(vol_parts) >= 4 and vol_parts[0].lower() == "volumes":
    _cat, _schema, _vol = vol_parts[1], vol_parts[2], vol_parts[3]
    spark.sql(f"CREATE VOLUME IF NOT EXISTS {_cat}.{_schema}.{_vol}")
    print(f"Volume {_cat}.{_schema}.{_vol} ensured.")

# Mock PEP (Politically Exposed Persons) data
pep_schema = StructType([
    StructField("id", LongType()),
    StructField("first_name", StringType()),
    StructField("last_name", StringType()),
    StructField("country", StringType()),
    StructField("date_of_birth", StringType()),
    StructField("position", StringType()),
    StructField("list_source", StringType()),
])
pep_rows = [
    Row(1, "Jan", "de Vries", "Netherlands", "1975-03-22", "Mayor", "National PEP"),
    Row(2, "Maria", "Garcia", "Spain", "1968-07-10", "Minister", "EU PEP"),
    Row(3, "John", "Smith", "United Kingdom", "1982-11-30", "MP", "UK PEP"),
    Row(4, "Erik", "van den Berg", "Netherlands", "1970-05-14", "Senator", "National PEP"),
]
spark.createDataFrame(pep_rows, pep_schema).write.mode("overwrite").saveAsTable(f"{CATALOG_SCHEMA}.pep_screening")

# Mock Sanctions data (no overlap with PEP)
sanctions_schema = StructType([
    StructField("id", LongType()),
    StructField("first_name", StringType()),
    StructField("last_name", StringType()),
    StructField("country", StringType()),
    StructField("date_of_birth", StringType()),
    StructField("sanction_type", StringType()),
    StructField("list_name", StringType()),
])
sanctions_rows = [
    Row(1, "Vladimir", "Petrov", "Russia", "1960-01-15", "Asset freeze", "OFAC"),
    Row(2, "Dimitri", "Kozlov", "Russia", "1972-08-20", "Travel ban", "EU Consolidated"),
    Row(3, "Anna", "Müller", "Germany", "1985-04-12", "Asset freeze", "UN List"),
    Row(4, "Pierre", "Dubois", "France", "1978-06-11", "Advisory", "National Watchlist"),
    Row(5, "Carlos", "Santos", "Brazil", "1965-09-05", "Asset freeze", "OFAC"),
]
spark.createDataFrame(sanctions_rows, sanctions_schema).write.mode("overwrite").saveAsTable(f"{CATALOG_SCHEMA}.sanctions_screening")

# Sanctioned countries list (OFAC, EU, UN comprehensive sanctions programmes)
sanctioned_countries_schema = StructType([
    StructField("country", StringType()),
    StructField("sanction_programme", StringType()),
    StructField("list_source", StringType()),
])
sanctioned_countries_rows = [
    Row("North Korea", "Comprehensive", "OFAC / EU / UN"),
    Row("Iran", "Comprehensive", "OFAC / EU / UN"),
    Row("Syria", "Comprehensive", "OFAC / EU"),
    Row("Cuba", "Comprehensive", "OFAC"),
    Row("Russia", "Comprehensive", "OFAC / EU"),
    Row("Belarus", "Targeted", "EU"),
    Row("Myanmar", "Targeted", "OFAC / EU"),
    Row("Venezuela", "Targeted", "OFAC"),
    Row("Afghanistan", "Targeted", "OFAC / UN"),
    Row("Somalia", "Targeted", "OFAC / UN"),
    Row("Yemen", "Targeted", "OFAC / UN"),
    Row("Libya", "Targeted", "UN"),
    Row("South Sudan", "Targeted", "EU / UN"),
]
spark.createDataFrame(sanctioned_countries_rows, sanctioned_countries_schema).write.mode("overwrite").saveAsTable(f"{CATALOG_SCHEMA}.sanctioned_countries")

# Mock shared fraud blacklist (customers and employees; multiple financial institutions)
fraud_schema = StructType([
    StructField("first_name", StringType()),
    StructField("last_name", StringType()),
    StructField("risk_score", IntegerType()),
    StructField("list_source", StringType()),
    StructField("role", StringType()),
])
fraud_rows = [
    Row("Jan", "Fraudster", 90, "Bank A", "customer"),
    Row("Maria", "Risky", 70, "Insurer B", "employee"),
    Row("Hans", "Betrüger", 85, "Bank C", "customer"),
    Row("Elena", "Kovac", 65, "Insurer A", "employee"),
]
spark.createDataFrame(fraud_rows, fraud_schema).write.mode("overwrite").saveAsTable(f"{CATALOG_SCHEMA}.fraud_list")

# Accounts table (populated by create_account tool in kyc_agent)
spark.sql(f"""
  CREATE TABLE IF NOT EXISTS {CATALOG_SCHEMA}.accounts (
    id STRING,
    account_type STRING,
    account_user STRING
  )
""")

# SBI risk codes: static table of ~100 CBS SBI codes (Standaard Bedrijfsindeling); risk_level for KYC SME check
# Source: CBS SBI structure (https://www.cbs.nl/nl-nl/onze-diensten/methoden/classificaties/activiteiten/standaard-bedrijfsindeling--sbi--/sbi-code-zoeken)
sbi_schema = StructType([
    StructField("sbi_code", StringType()),
    StructField("risk_level", StringType()),
    StructField("description", StringType()),
])
SBI_STATIC = [
    ("01", "low", "Crop and animal production, hunting"),
    ("01.11", "low", "Growing of cereals, legumes, oil seeds"),
    ("01.13", "low", "Growing of vegetables, melons, roots"),
    ("01.21", "low", "Growing of grapes"),
    ("01.41", "low", "Raising of dairy cattle"),
    ("01.49", "low", "Raising of other cattle"),
    ("02", "low", "Forestry and logging"),
    ("02.10", "low", "Silviculture and other forestry activities"),
    ("03", "low", "Fishing and aquaculture"),
    ("03.11", "low", "Marine fishing"),
    ("05", "medium", "Mining of coal and lignite"),
    ("06", "medium", "Extraction of crude petroleum and natural gas"),
    ("07", "medium", "Mining of metal ores"),
    ("08", "medium", "Other mining and quarrying"),
    ("10", "low", "Manufacture of food products"),
    ("10.11", "low", "Processing and preserving of meat"),
    ("10.12", "low", "Processing and preserving of poultry"),
    ("10.39", "low", "Processing of fruit and vegetables"),
    ("10.71", "low", "Manufacture of bread and fresh pastry"),
    ("11", "low", "Manufacture of beverages"),
    ("13", "low", "Manufacture of textiles"),
    ("14", "low", "Manufacture of wearing apparel"),
    ("15", "low", "Manufacture of leather and related products"),
    ("16", "low", "Manufacture of wood and wood products"),
    ("17", "low", "Manufacture of paper and paper products"),
    ("20", "low", "Manufacture of chemicals and chemical products"),
    ("21", "low", "Manufacture of pharmaceuticals"),
    ("22", "low", "Manufacture of rubber and plastic products"),
    ("23", "low", "Manufacture of non-metallic mineral products"),
    ("24", "low", "Manufacture of basic metals"),
    ("25", "low", "Manufacture of fabricated metal products"),
    ("26", "low", "Manufacture of computer, electronic and optical products"),
    ("27", "low", "Manufacture of electrical equipment"),
    ("28", "low", "Manufacture of machinery and equipment n.e.c."),
    ("29", "low", "Manufacture of motor vehicles and trailers"),
    ("30", "low", "Manufacture of other transport equipment"),
    ("31", "low", "Manufacture of furniture"),
    ("32", "low", "Other manufacturing"),
    ("33", "low", "Repair and installation of machinery"),
    ("35", "medium", "Electricity, gas, steam and air conditioning supply"),
    ("36", "low", "Water collection, treatment and supply"),
    ("37", "low", "Sewerage"),
    ("38", "low", "Waste collection, treatment and disposal"),
    ("39", "low", "Remediation and other waste management"),
    ("41", "low", "Construction of buildings"),
    ("42", "low", "Civil engineering"),
    ("43", "low", "Specialised construction activities"),
    ("45", "medium", "Wholesale and retail trade and repair of motor vehicles"),
    ("46", "medium", "Wholesale trade, except of motor vehicles"),
    ("46.31", "medium", "Wholesale of fruit and vegetables"),
    ("46.52", "medium", "Wholesale of electronic and telecommunications equipment"),
    ("47", "medium", "Retail trade, except of motor vehicles and motorcycles"),
    ("47.11", "medium", "Retail sale in non-specialised stores (food, beverages, tobacco)"),
    ("47.19", "medium", "Other retail sale in non-specialised stores"),
    ("47.24", "medium", "Retail sale of bread, pastry, confectionery"),
    ("47.41", "medium", "Retail sale of computers, peripheral units, software"),
    ("47.91", "medium", "Retail sale via mail order or Internet"),
    ("49", "medium", "Land transport and transport via pipelines"),
    ("49.41", "medium", "Freight transport by road"),
    ("50", "medium", "Water transport"),
    ("51", "medium", "Air transport"),
    ("52", "medium", "Warehousing and support activities for transportation"),
    ("53", "medium", "Postal and courier activities"),
    ("55", "medium", "Accommodation"),
    ("56", "medium", "Food and beverage service activities"),
    ("56.10", "medium", "Restaurants and mobile food service"),
    ("56.21", "medium", "Event catering"),
    ("58", "low", "Publishing activities"),
    ("59", "low", "Motion picture, video and television programme production"),
    ("60", "low", "Broadcasting and programming"),
    ("61", "low", "Telecommunications"),
    ("62", "low", "Computer programming, consultancy and related"),
    ("63", "low", "Information service activities"),
    ("64", "high", "Financial service activities, except insurance and pension funding"),
    ("64.11", "high", "Central banking"),
    ("64.19", "high", "Other monetary intermediation"),
    ("64.20", "high", "Activities of holding companies"),
    ("64.91", "high", "Financial leasing"),
    ("64.92", "high", "Other credit granting"),
    ("64.99", "high", "Other financial service activities n.e.c."),
    ("65", "high", "Insurance, reinsurance and pension funding"),
    ("65.11", "high", "Life insurance"),
    ("65.12", "high", "Non-life insurance"),
    ("65.20", "high", "Reinsurance"),
    ("66", "high", "Activities auxiliary to financial services and insurance"),
    ("66.11", "high", "Administration of financial markets"),
    ("66.19", "high", "Other activities auxiliary to financial services"),
    ("68", "medium", "Real estate activities"),
    ("68.20", "medium", "Renting and operating of own or leased real estate"),
    ("69", "low", "Legal and accounting activities"),
    ("70", "low", "Activities of head offices; management consultancy"),
    ("71", "low", "Architectural and engineering activities"),
    ("72", "low", "Scientific research and development"),
    ("73", "low", "Advertising and market research"),
    ("74", "low", "Other professional, scientific and technical activities"),
    ("77", "low", "Rental and leasing activities"),
    ("78", "low", "Employment activities"),
    ("79", "low", "Travel agency and tour operator activities"),
    ("80", "low", "Security and investigation activities"),
    ("81", "low", "Services to buildings and landscape activities"),
    ("82", "low", "Office administrative and business support activities"),
    ("84", "low", "Public administration and defence; compulsory social security"),
    ("85", "low", "Education"),
    ("85.10", "low", "Pre-primary education"),
    ("85.20", "low", "Primary education"),
    ("85.31", "low", "General secondary education"),
    ("85.59", "low", "Other education n.e.c."),
    ("86", "low", "Human health activities"),
    ("86.10", "low", "Hospital activities"),
    ("86.90", "low", "Other human health activities"),
    ("87", "low", "Residential care activities"),
    ("88", "low", "Social work activities without accommodation"),
    ("90", "low", "Creative, arts and entertainment activities"),
    ("91", "low", "Libraries, archives, museums and other cultural activities"),
    ("92", "high", "Gambling and betting activities"),
    ("93", "low", "Sports activities and amusement and recreation activities"),
    ("94", "low", "Activities of membership organisations"),
    ("95", "low", "Repair of computers and personal and household goods"),
    ("96", "low", "Other personal service activities"),
    ("97", "low", "Activities of households as employers"),
    ("99", "low", "Activities of extraterritorial organisations and bodies"),
]
sbi_rows = [Row(code, level, desc) for code, level, desc in SBI_STATIC]
spark.createDataFrame(sbi_rows, sbi_schema).write.mode("overwrite").saveAsTable(f"{CATALOG_SCHEMA}.sbi_risk_codes")

print(f"Created schema and tables in {CATALOG_SCHEMA}: pep_screening, sanctions_screening, sanctioned_countries, fraud_list, accounts, sbi_risk_codes ({len(SBI_STATIC)} CBS SBI codes)")

## 2. Create Unity Catalog functions

In [ ]:
# UC functions defined here (parameterized by CATALOG_SCHEMA); no external SQL file needed
cs = CATALOG_SCHEMA

check_pep_sql = f"""
CREATE OR REPLACE FUNCTION {cs}.check_pep(
  first_name STRING COMMENT 'First name from parsed KYC',
  last_name STRING COMMENT 'Last name from parsed KYC',
  country STRING COMMENT 'Country from parsed KYC',
  date_of_birth STRING COMMENT 'Date of birth from parsed KYC'
)
RETURNS TABLE (id LONG, first_name STRING, last_name STRING, country STRING, date_of_birth STRING, position STRING, list_source STRING)
COMMENT 'Check a person against the PEP list. Uses fuzzy name matching.'
RETURN
  SELECT p.id, p.first_name, p.last_name, p.country, p.date_of_birth, p.position, p.list_source
  FROM {cs}.pep_screening p
  WHERE levenshtein(lower(check_pep.first_name), lower(p.first_name)) <= 2
    AND levenshtein(lower(check_pep.last_name), lower(p.last_name)) <= 2
"""

check_sanctions_sql = f"""
CREATE OR REPLACE FUNCTION {cs}.check_sanctions(
  first_name STRING COMMENT 'First name from parsed KYC',
  last_name STRING COMMENT 'Last name from parsed KYC',
  country STRING COMMENT 'Country from parsed KYC',
  date_of_birth STRING COMMENT 'Date of birth from parsed KYC'
)
RETURNS TABLE (id LONG, first_name STRING, last_name STRING, country STRING, date_of_birth STRING, sanction_type STRING, list_name STRING)
COMMENT 'Check a person against the sanctions list. Uses fuzzy name matching.'
RETURN
  SELECT s.id, s.first_name, s.last_name, s.country, s.date_of_birth, s.sanction_type, s.list_name
  FROM {cs}.sanctions_screening s
  WHERE levenshtein(lower(check_sanctions.first_name), lower(s.first_name)) <= 2
    AND levenshtein(lower(check_sanctions.last_name), lower(s.last_name)) <= 2
"""

check_sanctioned_country_sql = f"""
CREATE OR REPLACE FUNCTION {cs}.check_sanctioned_country(
  country STRING COMMENT 'Country from parsed KYC (passport nationality)'
)
RETURNS TABLE (country STRING, sanction_programme STRING, list_source STRING)
COMMENT 'Check whether a country appears on the sanctioned countries list (OFAC, EU, UN). Uses case-insensitive exact match.'
RETURN
  SELECT sc.country, sc.sanction_programme, sc.list_source
  FROM {cs}.sanctioned_countries sc
  WHERE lower(trim(check_sanctioned_country.country)) = lower(trim(sc.country))
"""

check_customer_risk_sql = f"""
CREATE OR REPLACE FUNCTION {cs}.check_customer_risk(sbi_code STRING COMMENT 'SBI code of the business (e.g. 47.91)')
RETURNS TABLE (sbi_code STRING, risk_level STRING, description STRING)
COMMENT 'Check business SBI code against risk SBI codes. Valid range 01110-99000. If invalid or not in table, returns risk_level=failed.'
RETURN
  WITH input AS (
    SELECT trim(check_customer_risk.sbi_code) AS raw_code,
           try_cast(regexp_replace(regexp_replace(trim(check_customer_risk.sbi_code), '[.]', ''), ' ', '') AS INT) AS num_code
  ),
  valid_range AS (SELECT raw_code FROM input WHERE num_code BETWEEN 1110 AND 99000),
  matches AS (
    SELECT r.sbi_code, r.risk_level, r.description
    FROM {cs}.sbi_risk_codes r
    CROSS JOIN input i
    WHERE i.raw_code = r.sbi_code OR (length(r.sbi_code) <= length(i.raw_code) AND startswith(i.raw_code, r.sbi_code))
  )
  SELECT sbi_code, risk_level, description FROM matches
  UNION ALL
  SELECT i.raw_code, 'failed', 'SBI code not in risk table' FROM input i
  WHERE NOT EXISTS (SELECT 1 FROM matches) AND EXISTS (SELECT 1 FROM valid_range v WHERE v.raw_code = i.raw_code)
  UNION ALL
  SELECT i.raw_code, 'failed', 'SBI code out of valid range (01110-99000)' FROM input i
  WHERE NOT EXISTS (SELECT 1 FROM valid_range v WHERE v.raw_code = i.raw_code)
"""

check_fraud_sql = f"""
CREATE OR REPLACE FUNCTION {cs}.check_fraud(
  first_name STRING COMMENT 'First name from KYC',
  last_name STRING COMMENT 'Last name from KYC'
)
RETURNS TABLE (fraud_score INT, match_count INT, list_sources STRING)
COMMENT 'Check person against the shared fraud blacklist. Returns fraud_score, match_count, list_sources.'
RETURN
  SELECT coalesce(max(f.risk_score), 0) AS fraud_score, count(*) AS match_count,
         concat_ws(', ', collect_set(f.list_source)) AS list_sources
  FROM {cs}.fraud_list f
  WHERE levenshtein(lower(check_fraud.first_name), lower(f.first_name)) <= 2
    AND levenshtein(lower(check_fraud.last_name), lower(f.last_name)) <= 2
"""

for stmt in [check_pep_sql, check_sanctions_sql, check_sanctioned_country_sql, check_customer_risk_sql, check_fraud_sql]:
    spark.sql(stmt)
print(f"Created 5 UC functions in {CATALOG_SCHEMA}: check_pep, check_sanctions, check_sanctioned_country, check_customer_risk, check_fraud")

## 3. Generate demo document images

Generates synthetic passport and utility bill PNGs with **Pillow** and uploads them to the agent volume. Use the widgets to configure the persona — defaults match the built-in demo persona (Dirk van Dijk).

In [ ]:
dbutils.widgets.text("first_name",   "dirk",                       "First name")
dbutils.widgets.text("last_name",    "van_dijk",                   "Last name (use _ for spaces)")
dbutils.widgets.text("full_name",    "Dirk van Dijk",              "Display name")
dbutils.widgets.text("dob",          "15 JAN 1985",                "Date of birth (DD MON YYYY)")
dbutils.widgets.text("nationality",  "NETHERLANDS",                "Nationality")
dbutils.widgets.text("country_code", "NLD",                        "ISO country code")
dbutils.widgets.text("passport_no",  "NL7890123",                  "Passport number")
dbutils.widgets.text("issue_date",   "12 MAR 2020",                "Issue date")
dbutils.widgets.text("expiry_date",  "11 MAR 2030",                "Expiry date")
dbutils.widgets.text("address",      "123 Keizersgracht, Amsterdam, 1015 CJ", "Street address")
dbutils.widgets.text("bill_date",    "15 Jan 2025",                "Utility bill date")

In [ ]:
first_name   = dbutils.widgets.get("first_name").strip().lower()
last_name    = dbutils.widgets.get("last_name").strip().lower()
full_name    = dbutils.widgets.get("full_name").strip()
dob          = dbutils.widgets.get("dob").strip()
nationality  = dbutils.widgets.get("nationality").strip()
country_code = dbutils.widgets.get("country_code").strip()
passport_no  = dbutils.widgets.get("passport_no").strip()
issue_date   = dbutils.widgets.get("issue_date").strip()
expiry_date  = dbutils.widgets.get("expiry_date").strip()
address      = dbutils.widgets.get("address").strip()
bill_date    = dbutils.widgets.get("bill_date").strip()

passport_file    = f"passport_{first_name}_{last_name}.png"
utility_bill_file = f"utility_bill_{first_name}_{last_name}.png"

print(f"Persona  : {full_name}")
print(f"Passport : {passport_file}")
print(f"Utility  : {utility_bill_file}")
print(f"Volume   : {VOLUME_PATH}")

### 3a. Passport image


In [ ]:
from PIL import Image, ImageDraw, ImageFont
import io

# ── helpers ──────────────────────────────────────────────────────────────────

def _font(size):
    """Return a PIL font, falling back to the built-in bitmap font."""
    for face in [
        "/usr/share/fonts/truetype/liberation/LiberationMono-Regular.ttf",
        "/usr/share/fonts/truetype/dejavu/DejaVuSansMono.ttf",
        "/usr/share/fonts/truetype/freefont/FreeMono.ttf",
    ]:
        try:
            return ImageFont.truetype(face, size)
        except OSError:
            pass
    return ImageFont.load_default()


def _bold_font(size):
    for face in [
        "/usr/share/fonts/truetype/liberation/LiberationSans-Bold.ttf",
        "/usr/share/fonts/truetype/dejavu/DejaVuSans-Bold.ttf",
        "/usr/share/fonts/truetype/freefont/FreeSerifBold.ttf",
    ]:
        try:
            return ImageFont.truetype(face, size)
        except OSError:
            pass
    return ImageFont.load_default()


def _label(draw, x, y, text, font, color=(120, 120, 120)):
    draw.text((x, y), text, font=font, fill=color)


def _value(draw, x, y, text, font, color=(10, 10, 10)):
    draw.text((x, y), text, font=font, fill=color)


# ── passport builder ─────────────────────────────────────────────────────────

def make_passport(
    full_name, dob, nationality, country_code,
    passport_no, issue_date, expiry_date,
    width=900, height=640,
):
    img = Image.new("RGB", (width, height), color=(245, 242, 235))
    draw = ImageDraw.Draw(img)

    # Header bar
    draw.rectangle([(0, 0), (width, 80)], fill=(0, 51, 102))
    draw.text((20, 18), "PASPOORT  /  PASSPORT", font=_bold_font(28), fill=(255, 215, 0))
    draw.text((width - 200, 25), country_code, font=_bold_font(32), fill=(255, 215, 0))

    # Photo placeholder
    draw.rectangle([(30, 110), (210, 310)], fill=(200, 200, 200), outline=(150, 150, 150), width=2)
    draw.text((55, 195), "PHOTO", font=_bold_font(22), fill=(120, 120, 120))

    # Data fields — right of photo
    lf = _font(14)
    vf = _bold_font(18)

    name_parts = full_name.upper().split()
    surname    = name_parts[-1] if len(name_parts) > 1 else full_name.upper()
    given      = " ".join(name_parts[:-1]) if len(name_parts) > 1 else ""

    fields = [
        ("SURNAME / NOM",        surname),
        ("GIVEN NAMES / PRÉNOMS", given),
        ("NATIONALITY",           nationality),
        ("DATE OF BIRTH",         dob),
        ("PASSPORT NO.",          passport_no),
        ("DATE OF ISSUE",         issue_date),
        ("DATE OF EXPIRY",        expiry_date),
    ]

    x0, y0, row_h = 250, 100, 58
    for label, value in fields:
        _label(draw, x0, y0,      label, lf)
        _value(draw, x0, y0 + 18, value, vf)
        y0 += row_h

    # Divider
    draw.rectangle([(0, 320), (width, 323)], fill=(0, 51, 102))

    # Machine-readable zone (fake MRZ)
    mrz_font = _font(17)
    mrz_bg   = (225, 225, 215)
    draw.rectangle([(0, 323), (width, height)], fill=mrz_bg)
    draw.text((20, 330), "MACHINE READABLE ZONE (FICTITIOUS — FOR DEMO USE ONLY)",
              font=_font(11), fill=(160, 160, 160))

    last_mrz  = surname.replace(" ", "<").upper()
    first_mrz = given.replace(" ", "<").upper()
    mrz1 = f"P<{country_code:<3}{last_mrz}<<{first_mrz}".ljust(44, "<")[:44]
    mrz2 = f"{passport_no}<<<<<<{country_code}0000000M0000000<<<<<<<<<<<<<<<0"
    mrz2 = mrz2[:44]

    draw.text((20, 360), mrz1, font=mrz_font, fill=(30, 30, 30))
    draw.text((20, 395), mrz2, font=mrz_font, fill=(30, 30, 30))

    # Watermark
    draw.text((width - 320, height - 30), "SPECIMEN — NOT A REAL DOCUMENT",
              font=_font(12), fill=(180, 100, 100))

    return img


passport_img = make_passport(
    full_name    = full_name,
    dob          = dob,
    nationality  = nationality,
    country_code = country_code,
    passport_no  = passport_no,
    issue_date   = issue_date,
    expiry_date  = expiry_date,
)
passport_img

### 3b. Utility bill image


In [ ]:
import random

def make_utility_bill(
    full_name, address, bill_date,
    provider="Amsterdam Energy B.V.",
    account_no=None,
    amount_eur=None,
    width=820, height=1060,
):
    rng = random.Random(full_name + bill_date)  # deterministic per persona
    account_no = account_no or f"AMS-{rng.randint(100000, 999999)}"
    amount_eur = amount_eur or round(rng.uniform(55, 140), 2)

    img  = Image.new("RGB", (width, height), color=(255, 255, 255))
    draw = ImageDraw.Draw(img)

    # Company header
    draw.rectangle([(0, 0), (width, 90)], fill=(0, 102, 68))
    draw.text((30, 22), provider, font=_bold_font(30), fill=(255, 255, 255))
    draw.text((30, 60), "Energy & Utilities", font=_font(16), fill=(180, 230, 200))

    # Sub-header
    draw.rectangle([(0, 90), (width, 120)], fill=(230, 245, 235))
    draw.text((30, 98), "ENERGY INVOICE / ENERGIEREKENING",
              font=_bold_font(16), fill=(0, 80, 50))

    # Customer block
    draw.rectangle([(30, 140), (400, 280)], outline=(200, 200, 200), width=1)
    draw.text((40, 150), "Bill to / Factuuradres:", font=_font(13), fill=(120, 120, 120))
    draw.text((40, 172), full_name,                 font=_bold_font(18), fill=(20, 20, 20))
    # Split address on comma for multi-line
    addr_parts = [p.strip() for p in address.split(",")]
    y_addr = 198
    for part in addr_parts:
        draw.text((40, y_addr), part, font=_font(15), fill=(50, 50, 50))
        y_addr += 22

    # Invoice details block
    draw.rectangle([(450, 140), (width - 30, 280)], outline=(200, 200, 200), width=1)
    details = [
        ("Invoice date:",   bill_date),
        ("Account no.:",    account_no),
        ("Invoice no.:",    f"INV-{rng.randint(10000, 99999)}"),
        ("Due date:",       bill_date.replace("Jan", "Feb").replace("2025", "2025")),
    ]
    lf14 = _font(14)
    vf14 = _bold_font(14)
    y_det = 152
    for lbl, val in details:
        draw.text((460, y_det),       lbl, font=lf14, fill=(120, 120, 120))
        draw.text((600, y_det),       val, font=vf14, fill=(20, 20, 20))
        y_det += 26

    # Divider
    draw.rectangle([(30, 300), (width - 30, 302)], fill=(0, 102, 68))

    # Itemised usage table
    draw.text((30, 315), "Usage summary", font=_bold_font(16), fill=(0, 80, 50))
    headers = ["Description", "Unit", "Qty", "Rate", "Amount"]
    col_x   = [30, 310, 410, 510, 660]
    draw.rectangle([(30, 340), (width - 30, 365)], fill=(230, 245, 235))
    for h, x in zip(headers, col_x):
        draw.text((x, 345), h, font=_bold_font(14), fill=(0, 80, 50))

    rows = [
        ("Electricity supply",    "kWh", str(rng.randint(180, 320)),  "€0.28", f"€{round(rng.uniform(50.0, 90.0), 2):.2f}"),
        ("Gas supply",            "m³",  str(rng.randint(30, 80)),    "€1.12", f"€{round(rng.uniform(33.6, 89.6), 2):.2f}"),
        ("Network charges",       "-",   "1",                         "€8.50", "€8.50"),
        ("Energy tax (excl.)",    "-",   "1",                         "€3.80", "€3.80"),
    ]
    y_row = 370
    for i, (desc, unit, qty, rate, amt) in enumerate(rows):
        bg = (248, 252, 250) if i % 2 == 0 else (255, 255, 255)
        draw.rectangle([(30, y_row), (width - 30, y_row + 26)], fill=bg)
        for txt, x in zip([desc, unit, qty, rate, amt], col_x):
            draw.text((x, y_row + 5), txt, font=_font(14), fill=(40, 40, 40))
        y_row += 26

    # Total
    draw.rectangle([(30, y_row + 10), (width - 30, y_row + 12)], fill=(180, 180, 180))
    draw.rectangle([(450, y_row + 20), (width - 30, y_row + 52)], fill=(0, 102, 68))
    draw.text((460, y_row + 28), f"TOTAL DUE (incl. VAT):   €{amount_eur:.2f}",
              font=_bold_font(18), fill=(255, 255, 255))

    # Footer
    draw.rectangle([(0, height - 60), (width, height)], fill=(240, 240, 240))
    draw.text((30, height - 45),
              f"{provider}  |  KvK 12345678  |  BTW NL123456789B01  |  IBAN NL00 ABCD 0123 4567 89",
              font=_font(11), fill=(120, 120, 120))
    draw.text((width - 320, height - 25),
              "SPECIMEN — NOT A REAL DOCUMENT",
              font=_font(11), fill=(180, 100, 100))

    return img


bill_img = make_utility_bill(
    full_name = full_name,
    address   = address,
    bill_date = bill_date,
)
bill_img

### 3c. Upload to volume


In [ ]:
from databricks.sdk import WorkspaceClient

w = WorkspaceClient()

def _upload(img, filename):
    buf = io.BytesIO()
    img.save(buf, format="PNG")
    buf.seek(0)
    dest = VOLUME_PATH.rstrip("/") + "/" + filename
    w.files.upload(dest, buf, overwrite=True)
    print(f"Uploaded → {dest}")

_upload(passport_img,  passport_file)
_upload(bill_img, utility_bill_file)

### 3d. Verify

Confirm both files are readable from the volume.

In [ ]:
from PIL import Image as _PIL

for fname in [passport_file, utility_bill_file]:
    path = VOLUME_PATH.rstrip("/") + "/" + fname
    data = w.files.download(path)
    img  = _PIL.open(io.BytesIO(data.contents.read()))
    print(f"{fname}: {img.size} {img.mode} — OK")